# Only the blocks the construction uses

`init.ipynb` animates the whole of $W_{qk}$ for both layers. Most of those cells are exactly zero
in the hand construction, so the interesting structure is a few percent of the picture. This
notebook is the same run, the same 490 steps, the same minibatch gradients — restricted to the
**support of the construction**.

### The support, measured not guessed

| block | slice | shape | nnz | what the construction puts there |
|---|---|---|---|---|
| L1 recency ramp | `[8:108, 8:108]` | 100x100 | 10000 | $-207.85\,(t_q-t_k)$, diagonal $-1.04{\times}10^{10}$ |
| L1 noise-key kill | `[:, 4:8]` | 108x4 | 432 | $-1.04{\times}10^{10}$ — never attend to a noise key |
| L2 noise kill | `[0:8, 0:8]` | 8x8 | 32 | $-5.196{\times}10^{6}$ in columns 4:8 |
| L2 induction match | `[0:8, 108:116]` | 8x8 | 12 | $10392.3\,I_4$ + $103.9$ noise trigger in column 4 |
| L2 predecessor fallback | `[108:116, 108:116]` | 8x8 | 4 | $519.6\,I_4$ |

That is the whole of it: $W_{qk}$ L1 has 10,432 non-zeros and L2 has 48, and every one falls in a
block above. Rows 0:8 x columns 8:108 of layer 1 are exactly zero, as is everything in layer 2
outside those three 8x8 corners.

### What is plotted

Per step, parameters on the top row and their gradients directly underneath, one column per
block. Underneath that, two scalar panels — including **how much of each matrix's energy sits
inside the construction's support**, which is the direct question this restriction raises: is GD
putting its mass where the construction does, or somewhere else entirely?

Gradients are the cached `minibatch` ones from `init.ipynb` — the exact direction Adam was handed
at each step. Nothing is recomputed here.

In [ ]:
import os, glob, json, math, time
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.colors import SymLogNorm
from IPython.display import Video, Image, display

V, VS, T = 8, 4, 100
D, IN2 = V + T, 2 * (V + T)
SD = D ** 0.5
ALPHA_REC, GAMMA, P1, P2, NF, BIG = 20.0, -50000.0, 1000.0, 50.0, 10.0, -1e9

# ---- the construction, for reference values -------------------------------------------
Wq1 = torch.eye(D) * SD
T1 = torch.zeros(D, D)
for k in range(VS, V):
    T1[k, :] = BIG
for tq in range(T):
    for tk in range(T):
        T1[V + tk, V + tq] = BIG if tq == tk else -ALPHA_REC * (tq - tk)
Wk1 = T1.t()
Wq2 = torch.zeros(D, IN2)
Wq2[:VS, :VS] = torch.eye(VS) * SD * P1
Wq2[:VS, D:D + VS] = torch.eye(VS) * SD * P2
for v in range(V):
    Wq2[VS, v] = SD * NF
Wk2 = torch.zeros(D, IN2)
Wk2[:V, D:D + V] = torch.eye(V)
for v in range(VS, V):
    Wk2[VS, v] = GAMMA
CQK = [ (Wq1.t() @ Wk1).numpy(), (Wq2.t() @ Wk2).numpy() ]

# ---- the construction's support: (name, layer, rslice, cslice, aspect, annotate) -------
BLOCKS = [
    ('L1 recency ramp\n[8:108, 8:108]',        0, slice(V, D),     slice(V, D),     'equal', False),
    ('L1 noise-key kill\n[:, 4:8]',            0, slice(0, D),     slice(VS, V),    'auto',  False),
    ('L2 noise kill\n[0:8, 0:8]',              1, slice(0, V),     slice(0, V),     'equal', True),
    ('L2 induction match\n[0:8, 108:116]',     1, slice(0, V),     slice(D, D + V), 'equal', True),
    ('L2 predecessor fallback\n[108:116, 108:116]', 1, slice(D, D + V), slice(D, D + V), 'equal', True),
]
WRATIO = [2.4, 0.55, 1.0, 1.0, 1.0]

def support_mask(li):
    '''Boolean mask of the construction's non-zero support for layer li.'''
    m = np.zeros(CQK[li].shape, dtype=bool)
    for _, l, rs, cs, _, _ in BLOCKS:
        if l == li:
            m[rs, cs] = True
    return m

MASK = [support_mask(0), support_mask(1)]
for li in (0, 1):
    nz = CQK[li] != 0
    print(f'W_qk layer{li+1}: construction nnz {nz.sum():6d}  '
          f'support cells {MASK[li].sum():6d} / {MASK[li].size}  '
          f'({MASK[li].sum()/MASK[li].size:.2%})   '
          f'every nnz inside the support: {bool((nz & ~MASK[li]).sum() == 0)}')

plt.rcParams.update({'figure.dpi': 100, 'savefig.dpi': 120, 'font.size': 11,
                     'axes.grid': False})
FIGDIR = 'artifacts/figures_relevant'
os.makedirs(FIGDIR, exist_ok=True)

In [ ]:
CELL  = 'wqkinit_opt-adam_lr-0.005_b-256_init-normal_ep-5'
RDIR  = os.path.join('runs', CELL)
ADIR  = os.path.join('artifacts', CELL)
THRS  = [0.6, 0.8, 1.0]
KIND  = 'minibatch'          # gradient cache written by init.ipynb

SNAP, GRAD, SMETA, MET = {}, {}, {}, {}
for t in THRS:
    SMETA[t] = json.load(open(os.path.join(ADIR, f'stepmeta_x_{t}.json')))
    MET[t]   = json.load(open(os.path.join(RDIR, f'metrics_combined_seed0_x_{t}.json')))
    n = SMETA[t]['rows_written']
    SNAP[t] = [np.load(os.path.join(ADIR, f'wqk1_steps_x_{t}.npy'), mmap_mode='r')[:n],
               np.load(os.path.join(ADIR, f'wqk2_steps_x_{t}.npy'), mmap_mode='r')[:n]]
    gp = [os.path.join(ADIR, f'gstep_{KIND}{i}_x_{t}.npy') for i in (1, 2)]
    missing = [p for p in gp if not os.path.exists(p)]
    if missing:
        raise FileNotFoundError(f'{missing} -- run init.ipynb section 1 first (it writes '
                                f'the gradient caches this notebook reuses)')
    GRAD[t] = [np.load(gp[0], mmap_mode='r')[:n], np.load(gp[1], mmap_mode='r')[:n]]
    print(f'alpha={t}: {n} rows   params {SNAP[t][0].shape} {SNAP[t][1].shape}   '
          f'grads {GRAD[t][0].shape} {GRAD[t][1].shape}')

COLORS = {t: c for t, c in zip(THRS, ['#1f77b4', '#d62728', '#2ca02c'])}
SPE = MET[THRS[0]]['steps_per_epoch']

def diag_off(block, n=VS):
    s = np.asarray(block)[:n, :n]
    dg = np.abs(np.diag(s)).mean()
    of = np.abs(s - np.diag(np.diag(s))).sum() / (s.size - n)
    return dg / of if of > 1e-30 else float('inf')

def sym(ax, A, title, aspect='equal', annotate=False, vfix=None, fs=7, blocks_h=(), blocks_v=()):
    A = np.asarray(A, dtype=np.float64)
    v = vfix if vfix is not None else (float(np.abs(A).max()) or 1e-30)
    im = ax.imshow(A, cmap='RdBu_r', interpolation='nearest', aspect=aspect,
                   norm=SymLogNorm(linthresh=max(v / 100, 1e-30), linscale=0.6,
                                   vmin=-v, vmax=v, base=10))
    if annotate and A.size <= 64:
        for i in range(A.shape[0]):
            for j in range(A.shape[1]):
                ax.text(j, i, f'{A[i, j]:.3g}' if A[i, j] != 0 else '.', ha='center',
                        va='center', fontsize=fs, color='#111111')
    for b in blocks_h:
        ax.axhline(b - .5, color='k', lw=0.8)
    for b in blocks_v:
        ax.axvline(b - .5, color='k', lw=0.8)
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(title, fontsize=11)
    return im

## 1. The target — the construction's own blocks

What GD would have to reach, restricted to the same five panels. The ramp panel is scaled from
its **off-diagonal** entries only: the diagonal is $-1.04{\times}10^{10}$ and a scale that
includes it renders the ramp invisible.

In [ ]:
fig = plt.figure(figsize=(26, 6.2))
gs = fig.add_gridspec(1, len(BLOCKS), width_ratios=WRATIO, wspace=0.16)
for k, (nm, li, rs, cs, asp, ann) in enumerate(BLOCKS):
    ax = fig.add_subplot(gs[0, k])
    B = CQK[li][rs, cs]
    if k == 0:                       # ramp: ignore the huge diagonal when picking the scale
        off = B[~np.eye(B.shape[0], dtype=bool)]
        vfix = float(np.abs(off).max())
    else:
        vfix = None
    sym(ax, B, f'construction\n{nm}', aspect=asp, annotate=ann, vfix=vfix, fs=6)
fig.suptitle('The construction restricted to its own support  '
             '(ramp scaled off-diagonal only)', fontsize=15)
fig.savefig(os.path.join(FIGDIR, 'construction_blocks.png'), bbox_inches='tight')
plt.show()

print('reference values')
print(f'  L1 ramp      : off-diagonal -207.846*(t_q-t_k), range '
      f'[{CQK[0][V:D, V:D][~np.eye(T, dtype=bool)].min():.4g}, '
      f'{CQK[0][V:D, V:D][~np.eye(T, dtype=bool)].max():.4g}]   diagonal '
      f'{CQK[0][V, V]:.4g}')
print(f'  L1 noise cols: {CQK[0][0, VS]:.6g}  (all 108x4 cells)')
print(f'  L2 noise kill: {CQK[1][0, VS]:.6g}  in columns 4:8, zero in 0:4')
print(f'  L2 induction : diag {CQK[1][0, D]:.6g}   noise trigger col 4 {CQK[1][0, D+VS]:.6g}   '
      f'diag/off = inf')
print(f'  L2 fallback  : diag {CQK[1][D, D]:.6g} on 0:4, zero on 4:8')

## 2. Where the energy actually sits

The construction's support is 89.4% of layer 1 but only 0.10% of layer 2. So for layer 2 the
question has teeth: if GD spread its mass uniformly, the support would hold ~0.10% of
$\|W_{qk}\|^2$. Anything well above that means GD is concentrating exactly where the construction
does; anything near it means the learned solution lives somewhere else.

The same fraction is plotted for the gradient, which answers the sharper question — is the loss
even *pushing* on those cells?

In [ ]:
def share(A, mask):
    A = np.asarray(A, dtype=np.float64)
    tot = float((A ** 2).sum())
    return float((A[mask] ** 2).sum() / tot) if tot > 0 else 0.0

fig, axes = plt.subplots(2, 3, figsize=(21, 8.6))
ax = axes.ravel()
for t in THRS:
    st = SMETA[t]['step']
    n = len(st)
    for li in (0, 1):
        ax[li].plot(st, [share(SNAP[t][li][s], MASK[li]) for s in range(n)],
                    color=COLORS[t], lw=1.3, label=f'alpha={t}')
        ax[li + 3].plot(st, [share(GRAD[t][li][s], MASK[li]) for s in range(n)],
                        color=COLORS[t], lw=1.3, label=f'alpha={t}')
    ax[2].plot(st, [diag_off(np.asarray(SNAP[t][1][s])[:V, D:D + V]) for s in range(n)],
               color=COLORS[t], lw=1.3, label=f'alpha={t}  W')
    ax[5].plot(st, [np.asarray(SNAP[t][1][s])[:V, VS:V].mean() for s in range(n)],
               color=COLORS[t], lw=1.3, label=f'alpha={t}')

base = [MASK[0].sum() / MASK[0].size, MASK[1].sum() / MASK[1].size]
titles = ['L1: share of ||W||^2 inside the support',
          'L2: share of ||W||^2 inside the support',
          'L2 induction block diag/off  (construction = inf)',
          'L1: share of ||g||^2 inside the support',
          'L2: share of ||g||^2 inside the support',
          'L2 noise-kill block mean  (construction = -5.2e6, i.e. strongly negative)']
for i, a in enumerate(ax):
    if i in (0, 1, 3, 4):
        a.axhline(base[i % 3 if i < 3 else (i - 3)], color='k', ls='--', lw=1.0,
                  label=f'uniform = {base[0 if i in (0,3) else 1]:.4f}')
        a.set_ylim(0, 1.02)
    if i == 5:
        a.axhline(0, color='k', lw=0.8)
    for e in range(SPE, len(SMETA[THRS[0]]['step']), SPE):
        a.axvline(e, color='k', lw=0.6, ls=':', alpha=0.5)
    a.set_xlabel('optimizer step  (0 = random init)')
    a.set_title(titles[i], fontsize=11)
    a.grid(alpha=0.25); a.legend(fontsize=7)
fig.suptitle('Is gradient descent working inside the construction\'s support?', fontsize=15)
fig.tight_layout()
fig.savefig(os.path.join(FIGDIR, 'support_share.png'), bbox_inches='tight')
plt.show()

print(f'{"alpha":>6s} {"step":>6s} {"W L1 in":>9s} {"W L2 in":>9s} {"g L1 in":>9s} '
      f'{"g L2 in":>9s} {"ind d/o":>8s} {"noise-kill mean":>16s}')
print(f'{"":>6s} {"uniform":>6s} {base[0]:9.4f} {base[1]:9.4f} {base[0]:9.4f} {base[1]:9.4f}')
for t in THRS:
    for s in [0, 1, 10, 50, 98, 196, 490]:
        print(f'{t:6.1f} {s:6d} {share(SNAP[t][0][s], MASK[0]):9.4f} '
              f'{share(SNAP[t][1][s], MASK[1]):9.4f} {share(GRAD[t][0][s], MASK[0]):9.4f} '
              f'{share(GRAD[t][1][s], MASK[1]):9.4f} '
              f'{diag_off(np.asarray(SNAP[t][1][s])[:V, D:D+V]):8.2f} '
              f'{np.asarray(SNAP[t][1][s])[:V, VS:V].mean():16.4g}')
    print()

## 3. The animation — five blocks, parameters over gradients

Same clock as `init.ipynb`: one frame per optimizer step, 491 frames, row 0 the random init.
Parameters use a scale fixed across the window; gradients rescale per frame. The two 8x8 L2
blocks that matter carry their diag/off and mean in the panel titles.

In [ ]:
N_STEPS  = None       # None = all 490
STRIDE   = 1
FPS      = 15
ANIM_DPI = 100

def block_animation(t):
    n = SNAP[t][0].shape[0] if N_STEPS is None else min(N_STEPS + 1, SNAP[t][0].shape[0])
    rows = list(range(0, n, STRIDE))
    if rows[-1] != n - 1:
        rows.append(n - 1)
    st, h = SMETA[t]['step'], MET[t]['history']

    Wb = [np.asarray(SNAP[t][l][:n], dtype=np.float32)[:, rs, cs]
          for (_, l, rs, cs, _, _) in BLOCKS]
    Gb = [np.asarray(GRAD[t][l][:n], dtype=np.float32)[:, rs, cs]
          for (_, l, rs, cs, _, _) in BLOCKS]
    vW = [float(np.abs(b).max()) or 1e-30 for b in Wb]

    fig = plt.figure(figsize=(26, 14))
    gs = fig.add_gridspec(3, len(BLOCKS), height_ratios=[3.0, 3.0, 1.5],
                          width_ratios=WRATIO, hspace=0.22, wspace=0.16)
    axP = [fig.add_subplot(gs[0, k]) for k in range(len(BLOCKS))]
    axG = [fig.add_subplot(gs[1, k]) for k in range(len(BLOCKS))]
    axL = fig.add_subplot(gs[2, :2])
    axS = fig.add_subplot(gs[2, 2:])

    shW = {li: [share(SNAP[t][li][s], MASK[li]) for s in range(n)] for li in (0, 1)}
    shG = {li: [share(GRAD[t][li][s], MASK[li]) for s in range(n)] for li in (0, 1)}

    def extra(k, A):
        if k == 3:
            return f'   diag/off {diag_off(A):.2f}'
        if k == 2:
            return f'   mean {A[:, VS:V].mean():.3g}'
        return ''

    def draw(fi):
        s = rows[fi]
        for k, (nm, li, rs, cs, asp, ann) in enumerate(BLOCKS):
            for ax, arr, vfix, tag in ((axP[k], Wb[k][s], vW[k], 'W'),
                                       (axG[k], Gb[k][s], None, 'dL/dW')):
                ax.clear()
                sym(ax, arr, f'{tag}   {nm}\n|max| {np.abs(arr).max():.4g}{extra(k, arr)}',
                    aspect=asp, annotate=ann, vfix=vfix, fs=6)
        axL.clear()
        axL.plot(h['step'][1:], h['step_loss'], color='#444444', lw=0.9)
        axL.axvline(s, color='#d62728', lw=1.5)
        for e in range(SPE, n, SPE):
            axL.axvline(e, color='k', lw=0.6, ls=':', alpha=0.5)
        axL.set_xlabel('optimizer step'); axL.set_title('batch loss', fontsize=11)
        axL.grid(alpha=0.25)

        axS.clear()
        axS.plot(st[:n], shW[0], color='#1f77b4', lw=1.2, label='W  L1 in support')
        axS.plot(st[:n], shW[1], color='#1f77b4', lw=1.2, ls='--', label='W  L2 in support')
        axS.plot(st[:n], shG[0], color='#d62728', lw=1.2, label='g  L1 in support')
        axS.plot(st[:n], shG[1], color='#d62728', lw=1.2, ls='--', label='g  L2 in support')
        axS.axhline(MASK[1].sum() / MASK[1].size, color='k', ls=':', lw=1.0,
                    label='L2 uniform')
        axS.axvline(s, color='#d62728', lw=1.5)
        axS.set_ylim(0, 1.02); axS.set_xlabel('optimizer step')
        axS.set_title('fraction of squared norm inside the construction support', fontsize=11)
        axS.grid(alpha=0.25); axS.legend(fontsize=7, ncol=2, loc='upper left')

        ep = 0 if s == 0 else (s - 1) // SPE + 1
        head = ('random initialization, before any step' if s == 0
                else f'step {s}   (epoch {ep}, batch {(s - 1) % SPE})')
        fig.suptitle(f'alpha={t}   —   {head}   —   construction support only',
                     fontsize=18, y=0.985)

    anim = animation.FuncAnimation(fig, draw, frames=len(rows), interval=1000 // FPS,
                                   blit=False)
    out = os.path.join(FIGDIR, f'relevant_blocks_a{t}.mp4')
    try:
        anim.save(out, writer=animation.FFMpegWriter(fps=FPS, bitrate=8000), dpi=ANIM_DPI)
    except Exception as err:
        print(f'  ffmpeg unavailable ({type(err).__name__}: {err}) -- gif instead')
        out = out.replace('.mp4', '.gif')
        anim.save(out, writer=animation.PillowWriter(fps=FPS), dpi=ANIM_DPI)
    plt.close(fig)
    print(f'alpha={t}: {len(rows)} frames -> {out}')
    return out

for t in THRS:
    o = block_animation(t)
    display(Video(o, embed=True, width=1250) if o.endswith('.mp4') else Image(o))

## 4. The opening steps as stills, values printed

The three 8x8 layer-2 blocks with every cell value, parameters and gradients, over the first
steps — the form in which the induction diagonal would actually be readable if it were forming.

In [ ]:
STILL_STEPS = [0, 1, 2, 5, 10, 25, 50, 98]

for t in THRS:
    steps = [s for s in STILL_STEPS if s < SNAP[t][0].shape[0]]
    for src, tag in ((SNAP[t], 'W'), (GRAD[t], 'dL/dW')):
        sel = [(nm, li, rs, cs) for (nm, li, rs, cs, _, ann) in BLOCKS if ann]
        fig, axes = plt.subplots(len(sel), len(steps),
                                 figsize=(3.5 * len(steps), 3.7 * len(sel)))
        for r, (nm, li, rs, cs) in enumerate(sel):
            for k, s in enumerate(steps):
                A = np.asarray(src[li][s], dtype=np.float64)[rs, cs]
                sym(axes[r][k], A,
                    (f'{"init" if s == 0 else f"step {s}"}' if r == 0 else '') +
                    (f'\ndiag/off {diag_off(A):.2f}' if 'induction' in nm else ''),
                    annotate=True, fs=6)
                if k == 0:
                    axes[r][k].set_ylabel(nm.split('\n')[0], fontsize=10)
        fig.suptitle(f'alpha={t}   {tag} on the layer-2 blocks over the opening steps',
                     fontsize=15)
        fig.tight_layout()
        fig.savefig(os.path.join(FIGDIR,
                    f'stills_{"param" if tag == "W" else "grad"}_a{t}.png'),
                    bbox_inches='tight')
        plt.show()

## 5. What to look for

* **Layer 2's support share against the 0.10% baseline.** Section 2's dashed line is what a
  uniformly-spread matrix would give. If the learned share sits near it, the 48 cells the
  construction cares about are not where this solution lives — and the induction reading of the
  circuit does not survive.
* **The gradient's share vs the parameters'.** If the gradient concentrates on the support while
  the parameters do not, GD is being pushed the right way and simply has not travelled far in
  490 steps. If the gradient does not concentrate either, the loss is not asking for the
  construction at all.
* **The noise-kill block.** The construction puts $-5.2{\times}10^6$ in columns 4:8 and zero in
  0:4. The panel's printed mean should go negative early if the noise filter is being learned —
  and the epoch-level run's `signal_mass` climbing to 0.997 says something like it is.
* **The induction diagonal.** Section 4 prints every cell. The construction is $10392\,I_4$ plus a
  noise trigger; watch whether the four diagonal entries separate from their neighbours at all.
* **The ramp.** The construction's is smooth and antisymmetric about the diagonal. Even at a
  fraction of the amplitude, a learned ramp with the same ordering does the same job.